In [ ]:
import numpy as np
import pandas as pd
import warnings
import light_curve as lc
import lightgbm as lgb
import optuna

from tqdm import tqdm, TqdmWarning
from datetime import datetime
from pathlib import Path
from sklearn.metrics import f1_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from typing import Literal, Callable, Any


warnings.filterwarnings("ignore", category=TqdmWarning)

In [ ]:
Type = Literal["train", "test"]
Split = Literal["split_01", "split_02", "split_03", "split_04", "split_05", "split_06", "split_07", "split_08", "split_09", "split_10", "split_11", "split_12", "split_13", "split_14", "split_15", "split_16", "split_17", "split_18", "split_19", "split_20"]


EPS = np.finfo(float).eps


def now() -> str:
    return datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%z")

### Data Loading

In [ ]:
def load_log_df(type: Type, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{type}_log.parquet", **kwargs)

def load_flc_df(type: Type, split: Split, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet", **kwargs)


def __ingest_dfs(type: Type):
    log_df = pd.read_csv(f"../artifacts/kaggle/{type}_log.csv", index_col="object_id")
    log_df.to_parquet(f"../artifacts/kaggle/{type}_log.parquet")

    splits = sorted(log_df["split"].unique())
    for split in splits:
        flc_df = pd.read_csv(f"../artifacts/kaggle/{split}/{type}_full_lightcurves.csv")
        flc_df.to_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet")


__ingest_dfs(type="train")
__ingest_dfs(type="test")

### Feature Engineering

In [ ]:
def load_all_feats_df(type: Type, **kwargs):
    return pd.concat([pd.read_parquet(filepath, **kwargs) for filepath in Path("../artifacts/feats").rglob(f"{type}_feats.parquet")])

def load_feats_df(type: Type, split: Split, **kwargs):
    return pd.read_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet", **kwargs)


def __build_and_ingest_feats(type: Type):
    log_df = load_log_df(type=type)

    with tqdm(total=len(log_df), unit="obj") as pb:
        for split, log_sub_df in log_df.groupby("split"):
            pb.set_description(f"Building features for `{split}` (`{type}`)")

            feats_buf = []

            for obj_id, log_row in log_sub_df.iterrows():
                flc_df = load_flc_df(type=type, split=split, filters=[("object_id", "==", obj_id)]) # pyright: ignore[reportArgumentType]
                feats = __build_feats_for_obj(log_row, flc_df)
                feats_buf.append(feats)

                pb.update()

            Path(f"../artifacts/feats/{split}").mkdir(parents=True, exist_ok=True)
            pd.DataFrame(feats_buf).to_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet")


# TODO Add features acquired from domain knowledge
def __build_feats_for_obj(log_row: pd.Series, flc_df: pd.DataFrame) -> dict:
    # Could reorder?
    __de_extinct(log_row, flc_df)

    flc_df["Flux_ratio"] = flc_df["Flux"] / flc_df["Flux_err"]

    feats = log_row.to_dict()

    feats.update(__build_stats_feats_for_obj(flc_df))
    feats.update(__build_lc_feats_for_obj(flc_df))
    feats.update(__build_domain_feats_for_obj(flc_df))

    return feats


def __build_stats_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    feats = {}

    pivot_flc_df = flc_df.pivot_table(index="Time (MJD)", columns="Filter", values=["Flux", "Flux_err", "Flux_ratio"])

    for agg_name, agg in __STATS_AGGS.items():
        for feat in ["Flux", "Flux_err", "Flux_ratio"]:
            feats[f"{feat}_{agg_name}"] = agg(flc_df[feat])
            
            for filter in __FILTERS:
                if filter in pivot_flc_df.columns:
                    feats[f"{feat}_{agg_name}_{filter}"] = agg(pivot_flc_df[(feat, filter)]) # pyright: ignore[reportCallIssue]

    return feats


__STATS_AGGS: dict[str, Callable[[pd.Series], Any]] = {
    "mean": np.mean,
    "std": np.std,
    "min": np.min,
    "max": np.max,
    "median": np.median,
    "q25": lambda feats: feats.quantile(0.25),
    "q75": lambda feats: feats.quantile(0.75),
}
__FILTERS = ["u", "g", "r", "i", "z", "y"]


def __build_lc_feats_for_obj(flc_df: pd.DataFrame) -> dict:   
    feats = {}

    def lc_fe(df: pd.DataFrame):
        return __LC_FE(df.index.to_numpy(dtype=np.float64), df["Flux"].to_numpy(), df["Flux_err"].to_numpy()) # pyright: ignore[reportCallIssue]

    # TODO Try optimizing
    flc_fin_df = flc_df[(np.isfinite(flc_df.index) & np.isfinite(flc_df["Flux"]) & np.isfinite(flc_df["Flux_err"]))]
    
    if len(flc_fin_df) >= __LC_FE_MIN_NROWS:
        lc_feats = lc_fe(flc_fin_df)

        for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
            feats[feat_name] = feat

    for filter in __FILTERS:
        flc_fin_sub_df = flc_fin_df.loc[flc_df["Filter"] == filter]

        if len(flc_fin_sub_df) >= __LC_FE_MIN_NROWS:
            lc_feats = lc_fe(flc_fin_sub_df)

            for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
                feats[f"{feat_name}_{filter}"] = feat

    return feats


def __build_domain_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    return {}


__LC_FE = lc.Extractor(
    lc.LinearFit(), # pyright: ignore[reportArgumentType]
    lc.StetsonK(), # pyright: ignore[reportArgumentType]
    lc.Amplitude(), # pyright: ignore[reportArgumentType]
    lc.BeyondNStd(), # pyright: ignore[reportArgumentType]
    lc.Skew(), # pyright: ignore[reportArgumentType]
    lc.Kurtosis(), # pyright: ignore[reportArgumentType]
)
__LC_FE_MIN_NROWS = 4


# TODO Consider extinction.fitzpatrick99
def __de_extinct(log_row: pd.Series, flc_sub_df: pd.DataFrame):
    r_λ = flc_sub_df["Filter"].map({
        "u": 4.81,
        "g": 3.64,
        "r": 2.70,
        "i": 2.06,
        "z": 1.58,
        "y": 1.31
    })

    c_λ = np.pow(10, 0.4 * r_λ * log_row["EBV"])

    flc_sub_df["Flux"] *= c_λ
    flc_sub_df["Flux_err"] *= c_λ


__build_and_ingest_feats(type="train")
__build_and_ingest_feats(type="test")

### Data Cleaning

In [ ]:
train_df = load_all_feats_df(type="train")

X = train_df.drop(columns=["SpecType", "English Translation", "split", "target"])
y = train_df["target"]

X

### Aggressive Hyperparameter Tuning with F1 Threshold Optimization

In [ ]:
def __objective(trial: optuna.Trial):
    # Hyperparameters Suggestion
    lgbm_params = {
        "objective": "binary",
        "metric": "average_precision",
        "boosting_type": "gbdt",
        "n_jobs": -1,
        "verbosity": -1,
        # "is_unbalance": True,
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", low=1.0, high=200.0),

        # Tree Structure
        "num_leaves": trial.suggest_int("num_leaves", low=32, high=256),
        "max_depth": trial.suggest_int("max_depth", low=6, high=16),
        "min_child_samples": trial.suggest_int("min_child_samples", low=10, high=100),

        # Learning Speed
        "learning_rate": trial.suggest_float("learning_rate", low=0.005, high=0.1, log=True), # Slower learning rate leads to better accuracy
        "n_estimators": 10000,

        # Regularization
        "reg_alpha": trial.suggest_float("reg_alpha", low=1e-4, high=10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", low=1e-4, high=10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", low=1e-6, high=1.0, log=True),

        # Sampling
        "subsample": trial.suggest_float("subsample", low=0.5, high=0.95), # Stochasticity helps generalization
        "subsample_freq": trial.suggest_int("subsample_freq", low=1, high=5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", low=0.5, high=0.95),
    }

    # Cross-Validation
    skf = StratifiedKFold(n_splits=10, random_state=44, shuffle=True)
    oof_preds = np.zeros(len(X))

    for (train_idx, val_idx) in skf.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        lgbm = lgb.LGBMClassifier(**lgbm_params)
        lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="f1", callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            # optuna.integration.LightGBMPruningCallback(trial, "f1"),
        ])

        oof_preds[val_idx] = lgbm.predict_proba(X_val)[:, 1] # type: ignore
    
    # Dynamic Threshold Tuning
    thresholds = np.linspace(0.01, 0.99, 2000)

    f1_scores = np.array([f1_score(y, (oof_preds > threshold).astype(int)) for threshold in thresholds])
    idx = f1_scores.argmax()

    trial.set_user_attr("best_threshold", thresholds[idx])
    return f1_scores[idx]


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(n_startup_trials=44, multivariate=True, seed=44))
study.optimize(__objective, n_trials=200, n_jobs=-1, show_progress_bar=True) # pyright: ignore[reportArgumentType]

print(f"Study concluded with: best F1 score `{study.best_value:.4f}`; best threshold `{study.best_trial.user_attrs['best_threshold']:.2f}`; best hyperparameters: `{study.best_params}`")

In [ ]:
lgbm_params = study.best_params
lgbm_params.update({
    "objective": "binary",
    "metric": "average_precision",
    "boosting_type": "gbdt",
    "n_jobs": -1,
    "verbosity": -1,
})

### Feature Importance Analysis

### Inference

In [ ]:
# __dirpath = Path("../artifacts/predictions")
# __dirpath.mkdir(parents=True, exist_ok=True)

# prediction_df.to_csv(f"{__dirpath}/submission-{now()}.csv", index=False)

In [ ]:
# TODO
# Assuming GB, no imputation, no scaling, only minimal cleaning.
# Plot f1 score changes over trials & hyperparams
# Plot feature importance